In [ ]:
!pip install joblib

In [ ]:
!pip install scikit-learn==1.4.2 --only-binary=:all:

In [ ]:
!pip install xgboost

In [ ]:
!pip install imblearn

In [ ]:
import boto3
import sagemaker

In [ ]:
role = sagemaker.get_execution_role()
region = boto3.session.Session().region_name


print(role)

In [ ]:
import os
import tarfile

MODEL_JOBLIB = 'model/best_model.pkl'
TAR_OUTPUT= 'model.tar.gz'

if not os.path.exists(MODEL_JOBLIB):
    raise FileNotFoundError(
        f'{MODEL_JOBLIB} tidak ditemukan.\n'
    )

with tarfile.open(TAR_OUTPUT, "w:gz") as tar:
    tar.add(MODEL_JOBLIB, arcname="best_model.pkl")

with tarfile.open(TAR_OUTPUT, 'r:gz') as tar:
    print('Isi model.tar.gz:')
    for m in tar.getmembers():
        print(f'  {m.name}  ({m.size:,} bytes)')

print(f'\n{TAR_OUTPUT} siap.')

In [ ]:
BUCKET_NAME = '<your-bucket-name>'  #Ganti dengan nama bucket S3 Anda
S3_KEY = 'credit-score/model/model.tar.gz'

s3 = boto3.client('s3')
s3.upload_file('model.tar.gz', BUCKET_NAME, S3_KEY)

MODEL_S3_URI = f's3://{BUCKET_NAME}/{S3_KEY}'

In [ ]:
import sys
sys.path.insert(0, 'serve')

import importlib
import inference
importlib.reload(inference)

from inference import model_fn, input_fn, predict_fn, output_fn

# Load model dari direktori lokal
model = model_fn('model')
print('model_fn OK:', type(model))

In [ ]:
from sagemaker.sklearn.model import SKLearnModel
ENDPOINT_NAME = 'credit-score-endpoint'
INSTANCE_TYPE = 'ml.m5.large'
FRAMEWORK_VERSION = '1.4-2'

sm_model = SKLearnModel(
    model_data = MODEL_S3_URI,
    role = role,
    entry_point = 'inference.py',
    source_dir = 'serve',
    framework_version = FRAMEWORK_VERSION,
)

print(f'Deploying endpoint: {ENDPOINT_NAME} (5-8 minutes)...')

predictor = sm_model.deploy(
    initial_instance_count = 1,
    instance_type = INSTANCE_TYPE,
    endpoint_name = ENDPOINT_NAME,
)

print(f'\nEndpoint aktif: {ENDPOINT_NAME}')

In [ ]:
import json
import boto3

ENDPOINT_NAME = 'credit-score-endpoint'

runtime = boto3.client('sagemaker-runtime', region_name=region)

payload = json.dumps({'instances': [[
    0, 30, 'Engineer', 3000.0, 3, 3, 10, 2, 10, 5, 5.0, 3,
    'Standard', 1000.0, 30.0, 'Yes', 50.0, 50.0,
    'Low_spent_Small_value_payments', 300.0, 120
]]})

response = runtime.invoke_endpoint(
    EndpointName = ENDPOINT_NAME,
    ContentType  = 'application/json',
    Accept = 'application/json',
    Body = payload,
)

result = json.loads(response['Body'].read().decode('utf-8'))
print('Response:', result)

In [ ]:
import boto3, time

ENDPOINT_NAME = 'credit-score-endpoint'
region = boto3.session.Session().region_name
sm_client = boto3.client('sagemaker', region_name=region)

try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print(f'Deleting {ENDPOINT_NAME}...')
except Exception as e:
    print(f'Delete trigger: {e}')

print('Menunggu endpoint terhapus sempurna...')
while True:
    try:
        status = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)['EndpointStatus']
        print(f'  Status: {status} — masih ada, tunggu 15 detik...')
        time.sleep(15)
    except sm_client.exceptions.ClientError:
        print('Endpoint sudah terhapus. Siap deploy ulang.')
        break

try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print(f'Endpoint config deleted.')
except Exception as e:
    print(f'Config: {e}')